# Chapter 2 - Lab 5: Financial News Agent with Evaluator-Optimizer

**Main integrated version:** `v1.0-main-ddg-evaluator-optimizer`  
**Updated:** `2026-09-11` · **Model:** `gpt-4.1-mini`

## Definition and use case

The **Evaluator-Optimizer** pattern uses two roles: a generator/searcher produces a candidate answer and an evaluator judges it. If the evaluation is unsuccessful, the feedback is sent back to the searcher and a new attempt is made. The loop stops on success or after a maximum number of iterations.

The original lab asks for recent financial news with three constraints: **Reuters source**, **last two days**, and relevance to a requested **company/topic/region**. It also demonstrates single-region and multi-region requests.

### Updated implementation

The original notebook used `WebSearchTool`. In our tests, hosted web search worked in general but repeatedly failed to discover Reuters pages, while DDG did discover them. Therefore this version changes only the retrieval layer and keeps the Evaluator-Optimizer pattern:

```text
User -> Searcher (gpt-4.1-mini) -> search_reuters() -> DDG -> Reuters evidence
                                           |
                                           v
                              deterministic Python checks
                                           |
                                           v
                                Evaluator (gpt-4.1-mini)
                              successful -> STOP
                              unsuccessful -> feedback -> retry
```

Python validates objective constraints (Reuters domain, date window, URL count). The evaluator handles semantic constraints such as relevance and, for the original two-region example, whether each region received the requested number of articles.


In [1]:
!pip install -U openai openai-agents ddgs -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 52.0 MB/s eta 0:00:00


In [2]:
import json, os, re, sys, importlib.metadata as im
from dataclasses import dataclass
from datetime import datetime, timedelta, date
from typing import Literal
from urllib.parse import urlsplit, urlunsplit

from ddgs import DDGS
from google.colab import userdata
from agents import Agent, Runner
from agents.decorators import tool

# API key stored in Colab Secrets as OPENAI_API_KEY.
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Dynamic date window used by the lab.
TODAY = datetime.now().date()
START_DATE = TODAY - timedelta(days=2)

print("Python:", sys.version)
print("openai:", im.version("openai"))
print("openai-agents:", im.version("openai-agents"))
print("ddgs:", im.version("ddgs"))
print("API key present:", bool(os.environ.get("OPENAI_API_KEY")))
print("Allowed date window:", START_DATE.isoformat(), "->", TODAY.isoformat())

# ---------- deterministic helpers ----------
DATE_RE = re.compile(r"(20\d{2}-\d{2}-\d{2})")
URL_RE = re.compile(r"https?://[^\s)\]>]+")

def normalize_reuters_url(url: str) -> str | None:
    """Accept only Reuters URLs and normalize them for comparison."""
    try:
        p = urlsplit(url)
    except Exception:
        return None
    host = (p.hostname or "").lower().rstrip(".")
    if not (host == "reuters.com" or host.endswith(".reuters.com")):
        return None
    if p.scheme not in {"http", "https"} or not p.path:
        return None
    return urlunsplit(("https", host, p.path.rstrip("/") + "/", "", ""))

def date_from_reuters_url(url: str) -> date | None:
    """Reuters normally embeds YYYY-MM-DD in the article URL."""
    m = DATE_RE.search(urlsplit(url).path)
    if not m:
        return None
    try:
        return date.fromisoformat(m.group(1))
    except ValueError:
        return None

def in_required_window(url: str) -> bool:
    d = date_from_reuters_url(url)
    return d is not None and START_DATE <= d <= TODAY

def extract_reuters_urls(text: str) -> list[str]:
    """Extract distinct Reuters URLs from the generated answer."""
    out, seen = [], set()
    for raw in URL_RE.findall(text):
        url = normalize_reuters_url(raw.rstrip(".,;"))
        if url and url not in seen:
            seen.add(url)
            out.append(url)
    return out

def requested_count(text: str, default: int = 5) -> int:
    """Detect N in requests such as 'latest 5 news' or '5 Reuters articles'."""
    patterns = [
        r"\blatest\s+(\d+)\b",
        r"\b(\d+)\s+(?:Reuters\s+)?(?:articles?|news(?:\s+items?)?|items?)\b",
    ]
    for pattern in patterns:
        m = re.search(pattern, text, re.I)
        if m:
            return int(m.group(1))
    return default

REGION_ALIASES = {
    "Eurozone": [r"\beurozone\b", r"\beuro zone\b", r"\beuro area\b"],
    "United States": [r"\bunited states\b", r"\bu\.?s\.?a?\b", r"\busa\b"],
    "Europe": [r"\beurope\b"],
    "Asia": [r"\basia\b"],
}

def detect_regions(text: str) -> list[str]:
    regions = [
        region for region, patterns in REGION_ALIASES.items()
        if any(re.search(p, text, re.I) for p in patterns)
    ]
    if "Eurozone" in regions and "Europe" in regions:
        regions.remove("Europe")
    return regions

@dataclass
class RequestProfile:
    count_per_group: int
    total_required: int
    regions: list[str]
    per_region: bool

    def instruction(self) -> str:
        if self.per_region and self.regions:
            return (
                f"Return {self.count_per_group} items for EACH region "
                f"({', '.join(self.regions)}), {self.total_required} items total, "
                "organized in separate sections by region."
            )
        if self.regions:
            return f"Return {self.total_required} items relevant to {', '.join(self.regions)}."
        return f"Return {self.total_required} items total."

def build_request_profile(text: str) -> RequestProfile:
    """Preserves the original lab's '5 for each region' example correctly."""
    n = requested_count(text)
    regions = detect_regions(text)
    per_region = bool(re.search(
        r"\b(?:for\s+each|each|per)\s+region\b|\bfor\s+each\s+of\b",
        text, re.I
    ))
    total = n * len(regions) if per_region and len(regions) > 1 else n
    return RequestProfile(n, total, regions, per_region)


Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
openai: 3.13.0
openai-agents: 0.22.2
ddgs: 9.16.0
API key present: True
Allowed date window: 2026-09-09 -> 2026-09-11


In [3]:
@tool
def search_reuters(query: str, max_results: int = 10) -> str:
    """Search recent Reuters articles through DDG and return validated evidence."""
    search_query = f"site:reuters.com {query}".strip()
    print("\n[DDG TOOL] query:", search_query)

    try:
        raw = DDGS().text(
            search_query,
            region="us-en",
            safesearch="off",
            timelimit="w",
            max_results=max(1, min(max_results, 20)),
        )
    except Exception as exc:
        print("[DDG TOOL] ERROR:", type(exc).__name__, exc)
        return json.dumps({"query": search_query, "results": []})

    results, seen = [], set()
    for item in raw or []:
        url = normalize_reuters_url(item.get("href") or item.get("url") or "")
        if not url or url in seen:
            continue

        article_date = date_from_reuters_url(url)
        if article_date is None or not (START_DATE <= article_date <= TODAY):
            continue

        seen.add(url)
        results.append({
            "title": item.get("title") or "",
            "url": url,
            "publication_date": article_date.isoformat(),
            "snippet": item.get("body") or item.get("snippet") or "",
        })

    print("[DDG TOOL] valid Reuters results in window:", len(results))
    for r in results:
        print(" ", r["publication_date"], "-", r["title"])
        print("   ", r["url"])

    return json.dumps({
        "query": search_query,
        "window_start": START_DATE.isoformat(),
        "window_end": TODAY.isoformat(),
        "results": results,
    }, ensure_ascii=False)

# ---------- Searcher / Generator ----------
SEARCHER_INSTRUCTIONS = f"""
You are a financial-news research agent.
Reuters discovery MUST be done with the local search_reuters tool.
Do not use memory as evidence and do not invent headlines, dates, summaries or URLs.

Allowed publication window: {START_DATE.isoformat()} through {TODAY.isoformat()}, inclusive.
Only direct Reuters URLs returned by the tool are valid evidence.

If the first search is insufficient, call the tool again with different company,
topic or region keywords. Search requested regions separately when useful.
Prefer recent relevant items and remove duplicates.

Stock quote pages, ETFs, index values and generic quote pages do not count as news.

For every final item provide:
1. headline
2. publication date
3. short summary based only on tool title/snippet
4. Publisher: Reuters
5. direct Reuters URL

For multi-region requests, use separate sections by region.
"""

web_news_searcher = Agent(
    name="web_news_searcher",
    model="gpt-4.1-mini",
    instructions=SEARCHER_INSTRUCTIONS,
    tools=[search_reuters],
)

# ---------- Evaluator / LLM-as-a-Judge ----------
@dataclass
class EvaluationFeedback:
    feedback: str
    score: Literal["successful", "unsuccessful"]

EVALUATOR_INSTRUCTIONS = f"""
You are a strict evaluator of a Reuters financial-news answer.
Allowed dates: {START_DATE.isoformat()} through {TODAY.isoformat()}, inclusive.

Successful only when all applicable requirements are met:
- requested number of items;
- if N items were requested for EACH region, N relevant items for every region;
- headline/date/summary/Publisher Reuters/direct Reuters URL for every item;
- relevance to requested company/topic/region;
- enough distinct validated Reuters URLs;
- no stock quote, ETF, index-value or generic quote pages.

Zero or too few results is always unsuccessful.
The score MUST be exactly "successful" or "unsuccessful".
When unsuccessful, give concise actionable feedback for the next search.
"""

news_evaluator = Agent(
    name="news_evaluator",
    model="gpt-4.1-mini",
    instructions=EVALUATOR_INSTRUCTIONS,
    output_type=EvaluationFeedback,
)


In [4]:
async def run_news_request(msg: str, max_iterations: int = 4) -> str:
    """Evaluator-Optimizer loop used by the interactive cell and the examples."""
    profile = build_request_profile(msg)
    target_count = profile.total_required

    print("User's request:", msg)
    print("\nRequest profile:", profile.instruction())
    print("Required total:", target_count)
    print("Date window:", START_DATE.isoformat(), "->", TODAY.isoformat())

    feedback = None
    latest_answer = ""
    best_answer = ""
    best_valid_count = -1
    approved_answer = None

    for iteration in range(1, max_iterations + 1):
        print(
            "\n\033[92m"
            f"************************** NEWS SEARCH {iteration} **************************"
            "\033[0m"
        )

        # Start every retry from the original request + evaluator feedback.
        # This avoids carrying unsupported claims from failed attempts.
        if feedback is None:
            search_prompt = (
                f"ORIGINAL USER REQUEST:\n{msg}\n\n"
                f"INTERPRETED REQUIREMENTS:\n{profile.instruction()}"
            )
        else:
            search_prompt = (
                f"ORIGINAL USER REQUEST:\n{msg}\n\n"
                f"INTERPRETED REQUIREMENTS:\n{profile.instruction()}\n\n"
                f"PREVIOUS EVALUATOR FEEDBACK:\n{feedback}\n\n"
                "Run NEW Reuters searches with search_reuters. "
                "Change the search strategy where necessary."
            )

        # 1) GENERATE / SEARCH
        search_result = await Runner.run(web_news_searcher, search_prompt)
        latest_answer = str(search_result.final_output)
        print("\nGENERATED ANSWER:\n")
        print(latest_answer)

        # 2) DETERMINISTIC VALIDATION
        urls = extract_reuters_urls(latest_answer)
        valid_urls = [u for u in urls if in_required_window(u)]

        print(
            "\n\033[93m"
            "************************** DETERMINISTIC VALIDATION **************************"
            "\033[0m"
        )
        print("Reuters URLs in answer:", len(urls))
        print("Valid Reuters URLs in date window:", len(valid_urls))
        for u in valid_urls:
            print("  VALID:", u)

        # Preserve the best candidate in case the loop never fully converges.
        if len(valid_urls) > best_valid_count:
            best_valid_count = len(valid_urls)
            best_answer = latest_answer

        # 3) LLM EVALUATION
        evaluator_input = (
            f"ORIGINAL REQUEST:\n{msg}\n\n"
            f"REQUEST PROFILE:\n{profile.instruction()}\n\n"
            f"GENERATED ANSWER:\n{latest_answer}\n\n"
            f"REQUIRED TOTAL: {target_count}\n"
            f"VALID REUTERS URL COUNT: {len(valid_urls)}\n"
            "VALID REUTERS URLS:\n"
            + ("\n".join(valid_urls) if valid_urls else "NONE")
        )

        print(
            "\n\033[92m"
            "************************** RUNNING EVALUATION **************************"
            "\033[0m"
        )
        eval_result = await Runner.run(news_evaluator, evaluator_input)
        result: EvaluationFeedback = eval_result.final_output

        # 4) HARD PYTHON GATE: the LLM cannot approve too few valid URLs.
        if len(valid_urls) < target_count:
            result.score = "unsuccessful"
            result.feedback = (
                f"Only {len(valid_urls)} valid Reuters URLs were present; "
                f"{target_count} are required. Search again with new Reuters queries."
            )

        print("Evaluator score:", result.score)
        print("Evaluator feedback:", result.feedback)

        # 5) STOP IMMEDIATELY ON SUCCESS.
        if result.score == "successful" and len(valid_urls) >= target_count:
            approved_answer = latest_answer
            best_answer = latest_answer
            best_valid_count = len(valid_urls)
            print("Evaluation successful ==> stopping iteration.")
            break

        # 6) OPTIMIZER: feedback drives the next search attempt.
        feedback = result.feedback

        if iteration == max_iterations:
            print("Reached max_iterations ==> stopping iteration.")

    # Never replace a good answer with a later worse iteration.
    final_answer = approved_answer or best_answer or latest_answer

    print(
        "\n\033[92m"
        "************************** FINAL NEWS SET **************************"
        "\033[0m"
    )
    if approved_answer is not None:
        print(f"Using APPROVED answer with {best_valid_count} valid Reuters URLs.\n")
    else:
        print(
            f"No answer was fully approved; returning BEST candidate with "
            f"{best_valid_count} valid Reuters URLs.\n"
        )

    print(final_answer)
    return final_answer

async def main() -> None:
    """Interactive Colab entry point."""
    await run_news_request(input("User's request: ").strip())

# In Colab/Jupyter use: await main()


## Interactive run and original examples

Run the interactive version with:

```python
await main()
```

Do **not** use `asyncio.run(main())` inside Colab/Jupyter.

The original lab examples are preserved below. The actual Reuters stories will change with the execution date.

### Example 1 — Eurozone

Original prompt:

> Give me the latest 5 news items in the Eurozone, and specify the source for each one.

### Example 2 — Eurozone + United States

Original prompt:

> Give me the latest 5 news items for each region: Eurozone and US. Specify the source for each one, and organize the news into separate paragraphs by region.

The request parser correctly interprets this as **5 + 5 = 10 required Reuters items**.

### Example 3 — NVIDIA

This additional example demonstrates the same pattern for a company. In testing, a broad first search could fail and the evaluator feedback caused the Searcher to try more specific NVIDIA queries until it obtained an approved result.

The key learning point remains the same: **Generator -> Evaluator -> feedback -> improved Generator**, with deterministic Python checks around the LLM.


In [ ]:
# Interactive:
# await main()

# Original Example 1:
# await run_news_request(
#     "Give me the latest 5 news items in the Eurozone, "
#     "and specify the source for each one."
# )

# Original Example 2:
# await run_news_request(
#     "Give me the latest 5 news items for each region: Eurozone and US. "
#     "Specify the source for each one, and organize the news into "
#     "separate paragraphs by region."
# )

# Additional validated company example:
# await run_news_request("How is NVIDIA?")


In [6]:
# Original Example 1:
await run_news_request(
     "Give me the latest 5 news items in the Eurozone, "
     "and specify the source for each one."
 )

User's request: Give me the latest 5 news items in the Eurozone, and specify the source for each one.

Request profile: Return 5 items relevant to Eurozone.
Required total: 5
Date window: 2026-09-09 -> 2026-09-11

************************** NEWS SEARCH 1 **************************

[DDG TOOL] query: site:reuters.com Eurozone
[DDG TOOL] valid Reuters results in window: 0

GENERATED ANSWER:

There are no recent news items from Reuters on the Eurozone within the specified date range of 2026-09-09 through 2026-09-11. Would you like me to broaden the search parameters or look for news on a related topic?

************************** DETERMINISTIC VALIDATION **************************
Reuters URLs in answer: 0
Valid Reuters URLs in date window: 0

************************** RUNNING EVALUATION **************************
Evaluator score: unsuccessful
Evaluator feedback: Only 0 valid Reuters URLs were present; 5 are required. Search again with new Reuters queries.

************************** NEW

'Here are the latest 5 news items related to the Eurozone:\n\n1. "ECB hikes rates as Iran war adds to inflation angst | Reuters"\n   - Date: 2026-09-10\n   - Summary: The European Central Bank raised interest rates for the second time this year to address inflation driven by higher energy costs from the Iran conflict.\n   - Publisher: Reuters\n   - URL: https://www.reuters.com/business/view-ecb-hikes-rates-iran-war-adds-inflation-angst-2026-09-10/\n\n2. "ECB raises interest rates to fight off inflation jump | Reuters"\n   - Date: 2026-09-10\n   - Summary: The ECB raised interest rates, lifted its 2026 economic growth projection to 0.9%, and expects inflation averaging 3.0% in 2026 and 2.5% in 2027.\n   - Publisher: Reuters\n   - URL: https://www.reuters.com/business/ecb-raises-interest-rates-fight-off-inflation-jump-2026-09-10/\n\n3. "ECB hikes interest rates as $100 oil revives inflation fears"\n   - Date: 2026-09-10\n   - Summary: Eurozone inflation rose to 3.3% in August, up from 2.